# 02 - Patient Specific setup for pediatric DCM no VAD linear patient 

This notebook creates the patient-specific calibration setup for the pediatric DCM no-VAD linear scenario. It loads the filtered variable groups from 01B, fills the fixed target values for the first patient, computes the derived cardiac output target, defines first-pass bounds for the active tunable coefficients, and saves a patient-specific configuration file for the sampling notebook.

- Load the variable groups from 01B
- Fill patient-specific target values
- Compute derived target CO
- Define lower/upper bounds for the 11 tunable coefficients
- Save one patient-specific config file

### Libraries and Loading Variable Groups 

In [2]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path("../01_Data")
processed_dir = data_dir / "processed_pediatric_noVAD_linear" / "variable_groups"

groups_path = processed_dir / "variable_groups_pediatric_noVAD_linear.json"

with open(groups_path, "r") as f:
    variable_groups = json.load(f)

fixed_patient_targets = variable_groups["fixed_patient_targets"]
derived_calibration_targets = variable_groups["derived_calibration_targets"]
stage_1_tunables = variable_groups["stage_1_tunables"]
primary_output_targets = variable_groups["primary_output_targets"]
target_output_mapping = variable_groups["target_output_mapping"]

print("Fixed patient targets:", fixed_patient_targets)
print("Derived calibration targets:", derived_calibration_targets)
print("Stage 1 tunables:", stage_1_tunables)
print("Primary outputs:", primary_output_targets)

Fixed patient targets: ['weight', 'BSA', 'hr', 'SAP', 'DAP', 'CVP', 'LAP', 'sPAP', 'dPAP', 'EDV_LV', 'ESV_LV', 'EDV_RV', 'ESV_RV']
Derived calibration targets: ['CO']
Stage 1 tunables: ['k_Vtot', 'k_Vsys', 'k_Vusv_sys', 'k_Vusv_sys_ven', 'k_Vusv_pulm_ven', 'k_Ctot', 'k_Csys', 'k_Rsysven', 'k_Rpulmart', 'k_ESP_LV', 'k_ESP_RV']
Primary outputs: ['LAP_real', 'RAP_real', 'SAP_real', 'DAP_real', 'sPAP_real', 'dPAP_real', 'EDV_LV_real', 'ESV_LV_real', 'EDV_RV_real', 'ESV_RV_real', 'CO_real']


### Patient-specific target values

From the ORIG_ParameterèPediatric_DCM.txt

In [7]:
patient_id = "pediatric_dcm_patient_01"

patient_targets = {
    "weight": 21.75,
    "BSA": 0.92,
    "hr": 112,
    "SAP": 99,
    "DAP": 60,
    "CVP": 10,
    "LAP": 19,
    "sPAP": 40,
    "dPAP": 22,
    "EDV_LV": 99,
    "ESV_LV": 76,
    "EDV_RV": 63,
    "ESV_RV": 40,
}

missing_targets = [v for v in fixed_patient_targets if v not in patient_targets]

if missing_targets:
    raise ValueError(f"Missing patient target values: {missing_targets}")

print("All fixed patient targets are present.")

All fixed patient targets are present.


Computing derived calibration target CO (in mL/min)

In [8]:
sv_lv = patient_targets["EDV_LV"] - patient_targets["ESV_LV"]
patient_targets["CO"] = sv_lv * patient_targets["hr"]

print("SV_LV:", sv_lv)
print("CO:", patient_targets["CO"])

SV_LV: 23
CO: 2576


### Defining bounds for the 11 tunables 

In [ ]:
stage_1_bounds = pd.DataFrame([
    ["k_Vtot", 67.5, 60.0, 85.0, "Total blood volume coefficient; pediatric range should be validated"],
    ["k_Vsys", 0.84, 0.75, 0.90, "Fraction of total blood volume in systemic circulation"],
    ["k_Vusv_sys", 0.84, 0.75, 0.90, "Fraction of unstressed volume in systemic circulation"],
    ["k_Vusv_sys_ven", 0.95, 0.90, 0.99, "Fraction of systemic unstressed volume assigned to venous compartment"],
    ["k_Vusv_pulm_ven", 0.90, 0.80, 0.97, "Fraction of pulmonary unstressed volume assigned to pulmonary compartment"],
    ["k_Ctot", 2.15, 1.72, 2.58, "±20% around baseline total vascular compliance coefficient"],
    ["k_Csys", 0.85, 0.75, 0.95, "Fraction of total vascular compliance assigned to systemic circulation"],
    ["k_Rsysven", 60/1000, 0.048, 0.072, "±20% around systemic venous resistance scaling coefficient"],
    ["k_Rpulmart", 2/3, 0.50, 0.80, "Fraction of total pulmonary resistance assigned to pulmonary arterial resistance"],
    ["k_ESP_LV", 0.90, 0.75, 1.05, "LV end-systolic pressure fraction of systemic systolic pressure"],
    ["k_ESP_RV", 0.90, 0.75, 1.05, "RV end-systolic pressure fraction of systolic pulmonary arterial pressure"],
], columns=["variable", "baseline", "lower_bound", "upper_bound", "bound_reason"])

stage_1_bounds

,variable,baseline,lower_bound,upper_bound,bound_reason
0,k_Vtot,67.500000,60.000,85.000,Total blood volume coefficient; pediatric rang...
1,k_Vsys,0.840000,0.750,0.900,Fraction of total blood volume in systemic cir...
2,k_Vusv_sys,0.840000,0.750,0.900,Fraction of unstressed volume in systemic circ...
3,k_Vusv_sys_ven,0.950000,0.900,0.990,Fraction of systemic unstressed volume assigne...
4,k_Vusv_pulm_ven,0.900000,0.800,0.970,Fraction of pulmonary unstressed volume assign...
5,k_Ctot,2.150000,1.720,2.580,±20% around baseline total vascular compliance...
6,k_Csys,0.850000,0.750,0.950,Fraction of total vascular compliance assigned...
7,k_Rsysven,0.060000,0.048,0.072,±20% around systemic venous resistance scaling...
8,k_Rpulmart,0.666667,0.500,0.800,Fraction of total pulmonary resistance assigne...
9,k_ESP_LV,0.900000,0.750,1.050,LV end-systolic pressure fraction of systemic ...


Validating that bounds match the 01B tunables 

In [10]:
bounds_vars = set(stage_1_bounds["variable"])
tunables_vars = set(stage_1_tunables)

missing_bounds = tunables_vars - bounds_vars
extra_bounds = bounds_vars - tunables_vars

print("Missing bounds:", missing_bounds)
print("Extra bounds:", extra_bounds)

if missing_bounds:
    raise ValueError(f"Missing bounds for: {missing_bounds}")

if extra_bounds:
    print("Warning: bounds contain variables not in stage_1_tunables:", extra_bounds)

Missing bounds: set()
Extra bounds: set()


Numeric validity

In [11]:
for col in ["baseline", "lower_bound", "upper_bound"]:
    stage_1_bounds[col] = pd.to_numeric(stage_1_bounds[col], errors="coerce")

if stage_1_bounds[["baseline", "lower_bound", "upper_bound"]].isna().any().any():
    display(stage_1_bounds[stage_1_bounds[["baseline", "lower_bound", "upper_bound"]].isna().any(axis=1)])
    raise ValueError("Some bounds are missing or non-numeric.")

invalid_bounds = stage_1_bounds[
    stage_1_bounds["lower_bound"] >= stage_1_bounds["upper_bound"]
]

if len(invalid_bounds) > 0:
    display(invalid_bounds)
    raise ValueError("Some lower bounds are greater than or equal to upper bounds.")

print("Bounds look valid.")

Bounds look valid.


### Patient target table

In [12]:
patient_targets_df = pd.DataFrame(
    list(patient_targets.items()),
    columns=["variable", "value"]
)

patient_targets_df

,variable,value
0,weight,21.75
1,BSA,0.92
2,hr,112.00
3,SAP,99.00
4,DAP,60.00
5,CVP,10.00
6,LAP,19.00
7,sPAP,40.00
8,dPAP,22.00
9,EDV_LV,99.00


### Patient Specific Config 

In [13]:
patient_config = {
    "patient_id": patient_id,
    "scenario": variable_groups["scenario"],
    "fixed_patient_targets": {
        var: patient_targets[var]
        for var in fixed_patient_targets
    },
    "derived_calibration_targets": {
        var: patient_targets[var]
        for var in derived_calibration_targets
    },
    "all_patient_targets": patient_targets,
    "stage_1_tunables": stage_1_bounds.to_dict(orient="records"),
    "primary_output_targets": primary_output_targets,
    "target_output_mapping": target_output_mapping,
}

patient_config

{'patient_id': 'pediatric_dcm_patient_01',
 'scenario': 'pediatric_DCM_noVAD_linear',
 'fixed_patient_targets': {'weight': 21.75,
  'BSA': 0.92,
  'hr': 112,
  'SAP': 99,
  'DAP': 60,
  'CVP': 10,
  'LAP': 19,
  'sPAP': 40,
  'dPAP': 22,
  'EDV_LV': 99,
  'ESV_LV': 76,
  'EDV_RV': 63,
  'ESV_RV': 40},
 'derived_calibration_targets': {'CO': 2576},
 'all_patient_targets': {'weight': 21.75,
  'BSA': 0.92,
  'hr': 112,
  'SAP': 99,
  'DAP': 60,
  'CVP': 10,
  'LAP': 19,
  'sPAP': 40,
  'dPAP': 22,
  'EDV_LV': 99,
  'ESV_LV': 76,
  'EDV_RV': 63,
  'ESV_RV': 40,
  'CO': 2576},
 'stage_1_tunables': [{'variable': 'k_Vtot',
   'baseline': 67.5,
   'lower_bound': 60.0,
   'upper_bound': 85.0,
   'bound_reason': 'Total blood volume coefficient; pediatric range should be validated'},
  {'variable': 'k_Vsys',
   'baseline': 0.84,
   'lower_bound': 0.75,
   'upper_bound': 0.9,
   'bound_reason': 'Fraction of total blood volume in systemic circulation'},
  {'variable': 'k_Vusv_sys',
   'baseline': 0.

### Saving Outputs 

In [14]:
output_dir = data_dir / "patient_configs" / patient_id
output_dir.mkdir(parents=True, exist_ok=True)

config_path = output_dir / f"{patient_id}_config.json"
bounds_path = output_dir / f"{patient_id}_stage_1_bounds.csv"
targets_path = output_dir / f"{patient_id}_targets.csv"

with open(config_path, "w") as f:
    json.dump(patient_config, f, indent=4)

stage_1_bounds.to_csv(bounds_path, index=False)
patient_targets_df.to_csv(targets_path, index=False)

print("Saved patient config to:", config_path)
print("Saved bounds to:", bounds_path)
print("Saved targets to:", targets_path)

Saved patient config to: ..\01_Data\patient_configs\pediatric_dcm_patient_01\pediatric_dcm_patient_01_config.json
Saved bounds to: ..\01_Data\patient_configs\pediatric_dcm_patient_01\pediatric_dcm_patient_01_stage_1_bounds.csv
Saved targets to: ..\01_Data\patient_configs\pediatric_dcm_patient_01\pediatric_dcm_patient_01_targets.csv
